# Remapping an unstructured mesh onto a lat/lon grid

Four ways to move a field off an icosahedral mesh and onto the raster a model works on,
compared on a synthetic sphere where the right answer is known.

The methods are two different questions, not four variations of one:

- **k nearest neighbours** (`k=1` is *nearest*) and **barycentric** evaluate the field *at a point*.
- **conservative** evaluates its *mean over an area*.

Which is right depends on whether the target is coarser than the mesh. Where several cells
fall inside one target cell, sampling one of them is aliasing, however good the interpolant;
where the target is finer there is nothing to average and only interpolation can help.

The logic lives in `scripts/compare_remap_accuracy.py` so that the script and this notebook
cannot drift apart; this adds the pictures.

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

# the module lives in the repo, next to this notebook rather than on the path,
# so find the checkout by walking up from wherever jupyter was started
MODULE = "compare_remap_accuracy.py"
here = pathlib.Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "scripts" / MODULE).is_file()), None)
if root is None:
    raise FileNotFoundError(
        f"could not find scripts/{MODULE} above {here}. Start jupyter inside the makani "
        "checkout, or set root to it by hand."
    )
sys.path.insert(0, str(root / "scripts"))

import compare_remap_accuracy as remap  # noqa: E402

plt.rcParams["figure.dpi"] = 110

## The source: a mesh, and a field on it

Cell centres are laid out as a Fibonacci sphere, which is near uniform in the way an
icosahedral mesh is. The field has a smooth part that any of these grids can represent and
a noisy part that none of them can -- which is the whole point. A smooth field flatters
interpolation; real fields are not smooth at the grid scale.

In [ ]:
N_CELLS = 10000
NOISE = 0.3

source_lat, source_lon = remap.fibonacci_mesh(N_CELLS)
source_xyz = remap.unit_vectors(source_lat, source_lon)
source_values = remap.analytic_field(source_lat, source_lon, noise=NOISE)
source_area = np.full(N_CELLS, 4.0 * np.pi / N_CELLS)

tree = remap.cKDTree(source_xyz)
triangulation = remap.build_triangulation(source_xyz)

print(f"{N_CELLS} cells, {len(triangulation[0])} triangles, field range "
      f"[{source_values.min():.2f}, {source_values.max():.2f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, values, title in zip(
    axes,
    [remap.analytic_field(source_lat, source_lon, noise=0.0), source_values],
    ["smooth part (what a coarse grid can represent)", f"as sampled (noise = {NOISE})"],
):
    art = ax.scatter(source_lon, source_lat, c=values, s=2, cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set(title=title, xlabel="longitude", ylabel="latitude", xlim=(0, 360), ylim=(-90, 90))
    fig.colorbar(art, ax=ax)
fig.tight_layout()

## The remapped fields

One target resolution, every method. `ratio` is how many mesh cells fall in a target cell on
average: above one the target is coarser than the data, below one it is finer.

In [ ]:
N_LAT = 24  # target has N_LAT x 2*N_LAT points

target_lat, target_lon = remap.equiangular_grid(N_LAT)
n_target = N_LAT * 2 * N_LAT
grid_lat, grid_lon = (a.ravel() for a in np.meshgrid(target_lat, target_lon, indexing="ij"))
target_xyz = remap.unit_vectors(grid_lat, grid_lon)
target_area = remap.cell_areas(target_lat, target_lon)
flat_of_cell = remap.assign_cells_to_target(source_lat, source_lon, target_lat, target_lon)

# two references: the field at the target point, and its mean over the target cell
truth_point = remap.analytic_field(grid_lat, grid_lon, noise=0.0)
smooth_source = remap.analytic_field(source_lat, source_lon, noise=0.0)
caught = np.bincount(flat_of_cell, weights=source_area, minlength=n_target)
summed = np.bincount(flat_of_cell, weights=source_area * smooth_source, minlength=n_target)
truth_cell = np.where(caught > 0.0, summed / np.clip(caught, 1e-30, None), truth_point)

print(f"target {N_LAT} x {2 * N_LAT} = {n_target} points, "
      f"{N_CELLS / n_target:.2f} mesh cells per target cell, "
      f"{(caught <= 0).mean():.1%} of target cells empty")

In [ ]:
def run_all(target_xyz, flat_of_cell, target_lat, target_lon):
    results, empty = {}, {}
    for k in (1, 3, 6, 12):
        results[f"knn k={k}"] = remap.remap_knn(tree, source_values, target_xyz, k=k)
    results["barycentric"] = remap.remap_barycentric(
        tree, source_values, target_xyz, triangulation=triangulation
    )
    out, fallback = remap.remap_conservative(
        tree, source_values, target_xyz,
        target=(target_lat, target_lon, flat_of_cell), source_area=source_area,
    )
    results["conservative"] = out
    empty["conservative"] = fallback
    return results, empty


results, empty = run_all(target_xyz, flat_of_cell, target_lat, target_lon)

In [ ]:
def show(fields, titles, vmin, vmax, cmap, suptitle):
    columns = 3
    rows = int(np.ceil(len(fields) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4.4 * columns, 2.3 * rows), squeeze=False)
    for ax, field, title in zip(axes.ravel(), fields, titles):
        art = ax.imshow(field.reshape(len(target_lat), len(target_lon)), origin="upper",
                        extent=[0, 360, -90, 90], cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_title(title, fontsize=9)
        fig.colorbar(art, ax=ax)
    for ax in axes.ravel()[len(fields):]:
        ax.axis("off")
    fig.suptitle(suptitle)
    fig.tight_layout()


show([truth_cell] + list(results.values()),
     ["truth (cell mean)"] + list(results),
     -2, 2, "RdBu_r", f"remapped onto {N_LAT} x {2 * N_LAT}")

## What each one got wrong

Against the cell mean, which is what a grid this coarse can actually represent. Sampling
methods keep the noise and show it as speckle; averaging methods do not.

In [ ]:
show([results[name] - truth_cell for name in results], list(results),
     -0.8, 0.8, "coolwarm", "error against the cell mean")

## The numbers

`smoothing` is the standard deviation of the result over the input's: below one means the
method removed variance, which is what averaging is for and what sampling cannot do.
`conserv` is the change in the area weighted integral.

In [ ]:
source_integral = float(source_area @ source_values)

print(f"  {'method':14s} {'rmse@pt':>9s} {'rmse@cell':>10s} {'conserv':>10s} {'smoothing':>10s} {'empty':>7s}")
for name, field in results.items():
    conservation = abs(float(target_area @ field) - source_integral) / (4 * np.pi * float(np.std(source_values)))
    share = f"{empty[name]:6.1%}" if name in empty else "-"
    print(f"  {name:14s} {np.sqrt(np.mean((field - truth_point) ** 2)):9.4f} "
          f"{np.sqrt(np.mean((field - truth_cell) ** 2)):10.4f} {conservation:10.2e} "
          f"{np.std(field) / np.std(source_values):10.3f} {share:>7s}")

## How k trades noise against detail

More neighbours means more smoothing. Against a smooth reference that keeps paying, which is
why RMSE alone does not choose a method: it rewards blurring. The `smoothing` curve is the
other half of the story.

In [ ]:
neighbours = [1, 2, 3, 4, 6, 8, 12, 16, 24]
errors = [np.sqrt(np.mean((remap.remap_knn(tree, source_values, target_xyz, k=k) - truth_cell) ** 2))
          for k in neighbours]
spreads = [np.std(remap.remap_knn(tree, source_values, target_xyz, k=k)) / np.std(source_values)
           for k in neighbours]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(neighbours, errors, "o-", label="rmse vs cell mean")
ax.axhline(np.sqrt(np.mean((results["conservative"] - truth_cell) ** 2)), ls="--", c="k",
           label="conservative")
ax.set(xlabel="k", ylabel="rmse")
twin = ax.twinx()
twin.plot(neighbours, spreads, "s-", color="tab:red", alpha=0.6)
twin.axhline(1.0, ls=":", c="tab:red", alpha=0.5)
twin.set_ylabel("std / source std", color="tab:red")
ax.legend(loc="upper right")
fig.tight_layout()

## Where the crossover sits

Sweeping the target resolution moves the ratio of mesh cells per target cell through one.
Left of the dashed line the target is coarser than the data and averaging has something to
average; right of it the target is finer, the conservative operator runs out of cells and
falls back to nearest, and interpolation is the only thing left.

In [ ]:
resolutions = [12, 18, 24, 36, 48, 72, 96]
tracked = ["knn k=1", "knn k=3", "knn k=12", "barycentric", "conservative"]
curves = {name: [] for name in tracked}
ratios, fallbacks = [], []

for n_lat in resolutions:
    t_lat, t_lon = remap.equiangular_grid(n_lat)
    n_t = n_lat * 2 * n_lat
    g_lat, g_lon = (a.ravel() for a in np.meshgrid(t_lat, t_lon, indexing="ij"))
    xyz = remap.unit_vectors(g_lat, g_lon)
    flat = remap.assign_cells_to_target(source_lat, source_lon, t_lat, t_lon)

    point = remap.analytic_field(g_lat, g_lon, noise=0.0)
    got = np.bincount(flat, weights=source_area, minlength=n_t)
    total = np.bincount(flat, weights=source_area * smooth_source, minlength=n_t)
    cell_truth = np.where(got > 0.0, total / np.clip(got, 1e-30, None), point)

    step, step_empty = run_all(xyz, flat, t_lat, t_lon)
    for name in tracked:
        curves[name].append(float(np.sqrt(np.mean((step[name] - cell_truth) ** 2))))
    ratios.append(N_CELLS / n_t)
    fallbacks.append(step_empty["conservative"])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for name in tracked:
    axes[0].plot(ratios, curves[name], "o-", label=name)
axes[0].axvline(1.0, ls="--", c="k", alpha=0.5)
axes[0].set(xscale="log", xlabel="mesh cells per target cell", ylabel="rmse vs cell mean")
axes[0].invert_xaxis()
axes[0].legend(fontsize=8)

axes[1].plot(ratios, np.array(fallbacks) * 100, "o-", color="tab:red")
axes[1].axvline(1.0, ls="--", c="k", alpha=0.5)
axes[1].set(xscale="log", xlabel="mesh cells per target cell",
            ylabel="target cells with no data (%)")
axes[1].invert_xaxis()
fig.tight_layout()